In [1]:
"""Kaggle Eoin A. Moore NBA player-season averages, 2018-19 through 2025-26.

Kaggle source:
https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores/data

Add the dataset as a Kaggle notebook input or download PlayerStatistics.csv locally.
Run: python extract_nba_season_averages_2018_2026.py [--input PATH] [--output-dir PATH]

Creates eight per-season CSV files, one combined CSV, and one ZIP archive.
Calculations include regular-season games only, exclude DNPs, aggregate traded
players into a single player-season, and calculate shooting percentages from
season TOTALS rather than averaging game percentages.
"""

from __future__ import annotations

import argparse
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd

SOURCE_URL = "https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores/data"
SEASON_WINDOWS = {
    "2018-19": ("2018-10-01", "2019-07-01"),
    "2019-20": ("2019-10-01", "2020-11-01"), # COVID: season ended in 2020
    "2020-21": ("2020-12-01", "2021-08-01"), # COVID: season began Dec 2020
    "2021-22": ("2021-10-01", "2022-07-01"),
    "2022-23": ("2022-10-01", "2023-07-01"),
    "2023-24": ("2023-10-01", "2024-07-01"),
    "2024-25": ("2024-10-01", "2025-07-01"),
    "2025-26": ("2025-10-01", "2026-07-01"),
}

# Kaggle source columns -> per-game average output columns.
STATS = {
    "points": "PTS",
    "reboundsTotal": "REB",
    "assists": "AST",
    "steals": "STL",
    "blocks": "BLK",
    "turnovers": "TOV",
    "foulsPersonal": "PF",
    "fieldGoalsMade": "FGM",
    "fieldGoalsAttempted": "FGA",
    "threePointersMade": "3PM",
    "threePointersAttempted": "3PA",
    "freeThrowsMade": "FTM",
    "freeThrowsAttempted": "FTA",
    "reboundsOffensive": "OREB",
    "reboundsDefensive": "DREB",
    "plusMinusPoints": "PLUS_MINUS",
}

# These fields are verified as present in descriptions of the Kaggle source.
RELEVANT_COLUMNS = set(STATS) | {
    "personId", "firstName", "lastName", "name", "playerName",
    "gameId", "gameDate", "gameType", "gameLabel", "gameSubLabel", "numMinutes", "comment",
    "playerteamName", "playerteamId", "playerteamAbbreviation", "playerteamTricode",
}
OUTPUT_COLUMNS = (
    ["PlayerID", "Player", "Season", "Team", "Teams", "GP", "MPG"]
    + list(STATS.values()) + ["FG%", "3P%", "FT%", "TS%"]
)


def locate_player_data(input_path: str | None = None) -> Path:
    if input_path is not None:
        p = Path(input_path).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f"PlayerStatistics.csv not found at {p}")
        return p
    roots = [Path("/kaggle/input"), Path.cwd()]
    candidates: list[Path] = []
    for root in roots:
        if root.is_dir():
            candidates.extend(root.rglob("PlayerStatistics.csv"))
    candidates = list(dict.fromkeys(p.resolve() for p in candidates))
    if not candidates:
        raise FileNotFoundError(
            "PlayerStatistics.csv was not found. In Kaggle: Add Input > search "
            "'eoinamoore/historical-nba-data-and-player-box-scores'. "
            "Locally: pass --input '/path/to/PlayerStatistics.csv'."
        )
    preferred = [p for p in candidates if "historical-nba-data-and-player-box-scores" in str(p).lower()]
    if preferred:
        return sorted(preferred)[0]
    if len(candidates) > 1:
        raise ValueError(f"Multiple PlayerStatistics.csv files found; use --input: {candidates}")
    return candidates[0]


def parse_minutes(s: pd.Series) -> pd.Series:
    """Kaggle's numMinutes uses MM:SS or MM.SS, NOT decimal fractions.

    Examples: '30:30' and '30.30' both mean 30.5 minutes.
    """
    t = s.astype("string").str.strip()
    m = t.str.extract(r"^(\d{1,3})(?:[:.](\d{1,2}))?(?:\.0+)?$", expand=True)
    mins = pd.to_numeric(m[0], errors="coerce")
    secs = pd.to_numeric(m[1], errors="coerce").fillna(0)
    return (mins + secs / 60).where(secs.lt(60))


def read_source(path: Path) -> pd.DataFrame:
    print(f"Reading {path}")
    df = pd.read_csv(
        path,
        usecols=lambda c: c in RELEVANT_COLUMNS,
        dtype={"personId": "string", "gameId": "string", "numMinutes": "string"},
        low_memory=False,
    )
    if "personId" not in df.columns or "gameId" not in df.columns:
        raise ValueError(f"Expected personId and gameId in {path.name}; found: {list(df.columns)}")

    # Some versions have game metadata in the player table, others in Games.csv.
    if "gameDate" not in df or "gameType" not in df or df["gameType"].isna().all():
        game_path = path.with_name("Games.csv")
        if not game_path.is_file():
            raise ValueError("gameDate/gameType absent in PlayerStatistics.csv; add Games.csv from the same Kaggle dataset")
        meta = pd.read_csv(game_path, dtype={"gameId": "string"}, low_memory=False)
        required = {"gameId", "gameDate", "gameType"}
        if not required.issubset(meta):
            raise ValueError(f"Missing game metadata in Games.csv: {required - set(meta)}")
        wanted = [c for c in ("gameId", "gameDate", "gameType", "gameLabel", "gameSubLabel") if c in meta]
        meta = meta[wanted].drop_duplicates("gameId")
        df = df.merge(meta, on="gameId", suffixes=("", "__games"), how="left", validate="many_to_one")
        for c in wanted:
            if c == "gameId":
                continue
            alt = f"{c}__games"
            if alt in df:
                df[c] = df[c].fillna(df.pop(alt)) if c in df else df.pop(alt)

    if "gameType" not in df or "gameDate" not in df:
        raise ValueError("gameType and gameDate must be supplied by PlayerStatistics.csv or Games.csv")
    return df


def make_averages(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    types = df["gameType"].astype("string").str.strip().str.casefold()
    regular = types.str.fullmatch(r"regular(?:[\s_-]*season)?", na=False)
    if not regular.any():
        raise ValueError(f"No regular-season gameType found. Values: {df['gameType'].value_counts().head(10).to_dict()}")
    df = df.loc[regular].copy()

    # Defensive exclusion of one-off events classified as regular by a vendor.
    # The NBA Cup Final and All-Star events do not count in official season stats.
    for col in ("gameLabel", "gameSubLabel"):
        if col in df.columns:
            names = df[col].astype("string").fillna("")
            forbidden = names.str.contains(r"all[ -]?star|rising stars|nba cup (?:championship|final)\b|in.season tournament final\b", case=False, regex=True)
            df = df.loc[~forbidden].copy()
    df = df.loc[~df["gameId"].astype("string").str.startswith("006", na=False)].copy()

    df["gameDate"] = pd.to_datetime(df["gameDate"], utc=True, errors="coerce", format="mixed").dt.tz_localize(None)
    df["Season"] = pd.Series(pd.NA, dtype="string", index=df.index)
    for label, (start, end) in SEASON_WINDOWS.items():
        mask = df["gameDate"].ge(pd.Timestamp(start)) & df["gameDate"].lt(pd.Timestamp(end))
        df.loc[mask, "Season"] = label
    df = df.loc[df["Season"].notna()].copy()
    if df.empty:
        raise ValueError("No rows in the requested 2018-19 to 2025-26 seasons")

    if "numMinutes" not in df:
        raise ValueError("numMinutes is missing; cannot identify DNPs or calculate MPG")
    df["minutes"] = parse_minutes(df["numMinutes"])
    bad_min = df["minutes"].isna().sum()
    if bad_min:
        print(f"Notice: {bad_min:,} rows have missing/invalid minutes; they will not count as games played")
    df = df.loc[df["minutes"].gt(0)].copy()

    # A player can appear on several rosters in one season. Each NBA game counts once.
    before = len(df)
    df = df.sort_values("gameDate").drop_duplicates(["personId", "gameId"], keep="last")
    if before != len(df):
        print(f"Removed {before - len(df):,} repeated player/game records")

    if {"firstName", "lastName"}.issubset(df):
        df["Player"] = (df["firstName"].astype("string").fillna("").str.strip() + " "
                        + df["lastName"].astype("string").fillna("").str.strip()).str.strip()
    elif "playerName" in df:
        df["Player"] = df["playerName"].astype("string")
    elif "name" in df:
        df["Player"] = df["name"].astype("string")
    else:
        raise ValueError("No player name fields found")

    team_field = next((c for c in ("playerteamAbbreviation", "playerteamTricode", "playerteamName", "playerteamId") if c in df), None)
    if team_field:
        df["team_value"] = df[team_field].astype("string").str.strip()
    else:
        df["team_value"] = pd.Series(pd.NA, index=df.index, dtype="string")

    missing_stats = []
    for source in STATS:
        if source not in df:
            missing_stats.append(source)
            df[source] = np.nan
        df[source] = pd.to_numeric(df[source], errors="coerce")
    if missing_stats:
        print("Metrics missing from this Kaggle version (left blank):", ", ".join(missing_stats))

    keys = ["personId", "Season"]
    grouped = df.groupby(keys, sort=False, dropna=False)
    main = grouped.agg(Player=("Player", "last"), GP=("gameId", "size"), MPG=("minutes", "mean")).reset_index()
    totals = grouped[list(STATS)].sum(min_count=1).reset_index()
    main = main.merge(totals, on=keys, how="left", validate="one_to_one")

    def all_teams(s: pd.Series) -> str:
        return "; ".join(dict.fromkeys(x for x in s.dropna().astype(str) if x and x.lower() != "nan"))

    team_frame = grouped["team_value"].agg(all_teams).reset_index(name="Teams")
    main = main.merge(team_frame, on=keys, how="left", validate="one_to_one")
    main["Team"] = main["Teams"].apply(lambda s: "TOT" if "; " in s else s)

    for out, made, attempted in (
        ("FG%", "fieldGoalsMade", "fieldGoalsAttempted"),
        ("3P%", "threePointersMade", "threePointersAttempted"),
        ("FT%", "freeThrowsMade", "freeThrowsAttempted"),
    ):
        main[out] = 100 * main[made] / main[attempted].replace(0, np.nan)
    main["TS%"] = 100 * main["points"] / (2 * (main["fieldGoalsAttempted"] + 0.44 * main["freeThrowsAttempted"]).replace(0, np.nan))

    for source, dest in STATS.items():
        main[dest] = main[source] / main["GP"]
    main = main.rename(columns={"personId": "PlayerID"})
    main = main[OUTPUT_COLUMNS]
    pergame_cols = ["MPG", *STATS.values(), "FG%", "3P%", "FT%", "TS%"]
    main[pergame_cols] = main[pergame_cols].round(2)
    order = list(SEASON_WINDOWS)
    main["Season"] = pd.Categorical(main["Season"], categories=order, ordered=True)
    main = main.sort_values(["Season", "PlayerID"], kind="stable").reset_index(drop=True)
    main["Season"] = main["Season"].astype("string")
    return main


def write_outputs(results: pd.DataFrame, output: Path) -> Path:
    output.mkdir(parents=True, exist_ok=True)
    season_dir = output / "seasons"
    season_dir.mkdir(exist_ok=True)
    empty = [season for season in SEASON_WINDOWS if not results["Season"].eq(season).any()]
    if empty:
        raise ValueError(f"The input lacks regular-season records for: {', '.join(empty)}. Download the latest Kaggle version.")
    paths = []
    for season in SEASON_WINDOWS:
        part = results.loc[results["Season"] == season]
        dest = season_dir / f"nba_player_averages_{season.replace('-', '_')}.csv"
        part.to_csv(dest, index=False)
        paths.append(dest)
        print(f"{season}: {len(part):,} players -> {dest}")
    combo = output / "nba_player_averages_2018_19_to_2025_26_combined.csv"
    results.to_csv(combo, index=False)
    paths.append(combo)
    archive = output / "nba_player_season_averages_2018_19_to_2025_26.zip"
    with ZipFile(archive, "w", ZIP_DEFLATED) as z:
        for file in paths:
            z.write(file, arcname=file.relative_to(output))
    print(f"Created {len(results):,} player-season rows across {results.PlayerID.nunique():,} unique players")
    print(f"Combined CSV: {combo}")
    print(f"ZIP: {archive}")
    return archive


def main(input_path: str | None = None, output_dir: str | None = None) -> Path:
    source = locate_player_data(input_path)
    output = Path(output_dir) if output_dir is not None else (Path("/kaggle/working/nba_averages") if Path("/kaggle/working").is_dir() else Path("nba_averages"))
    return write_outputs(make_averages(read_source(source)), output)



In [2]:
from IPython.display import FileLink, display
archive = main()
display(FileLink(str(archive)))


Reading /kaggle/input/datasets/eoinamoore/historical-nba-data-and-player-box-scores/PlayerStatistics.csv
Notice: 43,101 rows have missing/invalid minutes; they will not count as games played
2018-19: 526 players -> /kaggle/working/nba_averages/seasons/nba_player_averages_2018_19.csv
2019-20: 528 players -> /kaggle/working/nba_averages/seasons/nba_player_averages_2019_20.csv
2020-21: 540 players -> /kaggle/working/nba_averages/seasons/nba_player_averages_2020_21.csv
2021-22: 602 players -> /kaggle/working/nba_averages/seasons/nba_player_averages_2021_22.csv
2022-23: 539 players -> /kaggle/working/nba_averages/seasons/nba_player_averages_2022_23.csv
2023-24: 570 players -> /kaggle/working/nba_averages/seasons/nba_player_averages_2023_24.csv
2024-25: 569 players -> /kaggle/working/nba_averages/seasons/nba_player_averages_2024_25.csv
2025-26: 582 players -> /kaggle/working/nba_averages/seasons/nba_player_averages_2025_26.csv
Created 4,456 player-season rows across 1,273 unique players
Comb

/kaggle/working/nba_averages/nba_player_season_averages_2018_19_to_2025_26.zip